# Module 2 - Inference Pipeline Demo

This notebook demonstrates the hierarchical classification inference pipeline.

## Prerequisites

1. Run Module 1 (build_taxonomy) first to create taxonomy_index
2. Ensure taxonomy_index exists for the target taxonomy (e.g., ISCO)

## Approach

This notebook:
1. Creates a single session for the entire notebook
2. Tests each node individually first
3. Then tests the full pipeline
4. Shows batch inference with multiple queries

In [ ]:
import pandas as pd
import numpy as np
import json
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
from pathlib import Path

In [ ]:
# Bootstrap Kedro project
project_path = Path.cwd().parent
metadata = bootstrap_project(project_path)
print(f"Project: {metadata.project_name}")

# Create session and get context (reuse throughout notebook)
session = KedroSession.create(project_path=project_path)
context = session.load_context()

# Access catalog and parameters
catalog = context.catalog
params = context.params

print(f"Taxonomy key: {params['taxonomy_key']}")
print(f"Model: {params['model_name']}")

## Step 1: Verify Taxonomy Index Exists

In [ ]:
# Check if taxonomy index exists
taxonomy_key = params['taxonomy_key']
index_path = project_path / "data" / "03_primary" / "taxonomies" / "index" / f"{taxonomy_key}.parquet"

if index_path.exists():
    print(f"✓ Taxonomy index found: {index_path}")
    
    # Load and inspect
    taxonomy_df = pd.read_parquet(index_path)
    print(f"  - Nodes: {len(taxonomy_df)}")
    print(f"  - Levels: {sorted(taxonomy_df['level'].unique())}")
    print(f"  - Embedding dim: {taxonomy_df['embedding_dim'].iloc[0]}")
    print(f"  - Evidence nodes: {(taxonomy_df['evidence_count'] > 0).sum()}")
    print(f"  - Root nodes: {(taxonomy_df['parentCode'] == '__root__').sum()}")
else:
    print(f"✗ Taxonomy index NOT found: {index_path}")
    print("  Run: kedro run --pipeline=build_taxonomy")

## Step 2: Test Individual Nodes

### Node 1: Load Embedding Model

In [ ]:
# Test loading embedding model
from sentence_transformers import SentenceTransformer

model_name = params['model_name']
print(f"Loading model: {model_name}")

embedding_model = SentenceTransformer(model_name, trust_remote_code=True)
print(f"✓ Model loaded: {embedding_model.get_sentence_embedding_dimension()} dimensions")

# Save to catalog
catalog.save("inference_embedding_model", embedding_model)
print("✓ Saved to catalog")

### Node 2: Load Taxonomy Index

In [ ]:
# Test loading taxonomy index
from taxomind.pipelines.inference.nodes import load_taxonomy_index

taxonomy_index = catalog.load("taxonomy_index")
taxonomy_key = params['taxonomy_key']

inference_taxonomy_df = load_taxonomy_index(taxonomy_index, taxonomy_key)
print(f"✓ Loaded taxonomy: {len(inference_taxonomy_df)} nodes")
print(f"  Columns: {list(inference_taxonomy_df.columns)}")
print(f"  Sample codes: {inference_taxonomy_df['code'].head(5).tolist()}")

# Save to catalog
catalog.save("inference_taxonomy_df", inference_taxonomy_df)
print("✓ Saved to catalog")

### Node 3: Build Taxonomy Graph

In [ ]:
# Test building taxonomy graph
from taxomind.pipelines.inference.nodes import load_taxonomy_graph

taxonomy_graph = load_taxonomy_graph(inference_taxonomy_df)
print(f"✓ Built taxonomy graph: {len(taxonomy_graph)} parent nodes")
print(f"  Root children: {len(taxonomy_graph.get('__root__', []))}")
print(f"  Sample root codes: {taxonomy_graph.get('__root__', [])[:5]}")

# Save to catalog
catalog.save("inference_taxonomy_graph", taxonomy_graph)
print("✓ Saved to catalog")

### Node 4: Build Retrieval Index

In [ ]:
# Test building retrieval index
from taxomind.pipelines.inference.nodes import build_retrieval_index

retrieval_index = build_retrieval_index(inference_taxonomy_df)
print(f"✓ Built retrieval index:")
print(f"  Method: {retrieval_index['method']}")
print(f"  Embeddings shape: {retrieval_index['embeddings'].shape}")
print(f"  Codes: {len(retrieval_index['codes'])}")

# Save to catalog
catalog.save("inference_retrieval_index", retrieval_index)
print("✓ Saved to catalog")

### Node 5: Prepare Scoring Views

In [ ]:
# Test preparing scoring views
from taxomind.pipelines.inference.nodes import prepare_scoring_views

scoring_views = prepare_scoring_views(inference_taxonomy_df)
print(f"✓ Prepared scoring views:")
print(f"  Nodes tracked: {len(scoring_views['view_availability'])}")

# Count view availability
has_def = sum(1 for v in scoring_views['view_availability'].values() if v['has_definition'])
has_ex = sum(1 for v in scoring_views['view_availability'].values() if v['has_examples'])
has_ev = sum(1 for v in scoring_views['view_availability'].values() if v['has_evidence'])

print(f"  Nodes with definitions: {has_def}")
print(f"  Nodes with examples: {has_ex}")
print(f"  Nodes with evidence: {has_ev}")

# Save to catalog
catalog.save("inference_scoring_views", scoring_views)
print("✓ Saved to catalog")

### Node 6: Load Queries

In [ ]:
# Test loading queries (single query)
from taxomind.pipelines.inference.nodes import load_queries

query_text = "education"
print(f"Query: {query_text}")

queries_df = load_queries(query_text)
print(f"✓ Loaded queries: {len(queries_df)} query")
print(queries_df)

# Save to catalog
catalog.save("inference_queries_df", queries_df)
print("✓ Saved to catalog")

### Node 7: Embed Queries

In [ ]:
# Test embedding queries
from taxomind.pipelines.inference.nodes import embed_queries

batch_size = params['embedding']['batch_size']
queries_embedded_df = embed_queries(queries_df, embedding_model, batch_size)

print(f"✓ Embedded queries: {len(queries_embedded_df)} query")
print(f"  Embedding shape: {queries_embedded_df['embedding'].iloc[0].shape}")

# Save to catalog
catalog.save("inference_queries_embedded_df", queries_embedded_df)
print("✓ Saved to catalog")

### Node 8: Batch Inference

In [ ]:
# Test batch inference
from taxomind.pipelines.inference.nodes import batch_inference

predictions_df = batch_inference(
    queries_df=queries_embedded_df,
    retrieval_index=retrieval_index,
    scoring_views=scoring_views,
    taxonomy_graph=taxonomy_graph,
    taxonomy_df=inference_taxonomy_df,
    retrieval_k=params['inference']['retrieval_k'],
    min_descent_gap=params['inference']['min_descent_gap'],
    parent_veto_margin=params['inference']['parent_veto_margin'],
    beta=params['inference']['beta'],
    max_depth=params['inference']['max_depth'],
)

print(f"✓ Batch inference complete: {len(predictions_df)} prediction")
predictions_df

## Step 3: Inspect Prediction

Examine the prediction structure in detail.

In [ ]:
# Get first prediction
prediction = predictions_df.iloc[0]

print("=" * 80)
print("PREDICTION DETAILS")
print("=" * 80)

print(f"\nQuery: {prediction['query']}")
print(f"\nPredicted: {prediction['predicted_code']} - {prediction['predicted_label']}")
print(f"  Level: {prediction['predicted_level']}")
print(f"  Score: {prediction['score']:.3f}")

print(f"\nAmbiguous: {prediction['ambiguous']}")
print(f"Stopping Reason: {prediction['stopping_reason']}")

if prediction['alternatives']:
    print(f"\nAlternatives ({len(prediction['alternatives'])}):")
    for alt in prediction['alternatives']:
        print(f"  - {alt['code']}: {alt['label']} (score={alt['score']:.3f})")

if prediction['path']:
    print(f"\nPath (root to prediction):")
    print(" → ".join(prediction['path']))

## Step 4: Routing Trace Analysis

In [ ]:
print("=" * 80)
print("ROUTING TRACE")
print("=" * 80)

routing_trace = prediction['routing_trace']
for decision in routing_trace:
    print(f"\nLevel {decision['level']}:")
    print(f"  Candidates: {len(decision['candidates'])}")
    print(f"  Top scores:")
    for code, score in list(decision['scores'].items())[:3]:
        label = inference_taxonomy_df[inference_taxonomy_df['code'] == code]['label'].iloc[0]
        print(f"    - {code} ({label}): {score:.3f}")
    print(f"  Selected: {decision['selected']}")
    print(f"  Stopped: {decision['stopped']}")
    print(f"  Reason: {decision['reason']}")

## Step 5: Test Full Pipeline (Single Query)

Now test the complete pipeline end-to-end with a single query.

In [ ]:
# Test full pipeline with a single query
query = "I teach mathematics at a university"
print(f"Testing pipeline with: {query}\n")

# Save query input
catalog.save("inference_query_input", query)

# Run pipeline
session.run(
    pipeline_name="inference",
    from_inputs=["inference_query_input"],
)

# Load results
result_df = catalog.load("inference_predictions_df")
print(f"\n✓ Pipeline completed successfully")
result_df

## Step 6: Batch Inference (Multiple Queries)

Process multiple queries at once for efficiency.

In [ ]:
test_queries = [
    "I work as a software developer",
    "I work in education",
    "I teach at university",
    "I manage a retail store",
    "I work with computers",
]

print(f"Processing {len(test_queries)} queries in batch...\n")

# Save batch query input
catalog.save("inference_query_input", test_queries)

# Run pipeline
session.run(
    pipeline_name="inference",
    from_inputs=["inference_query_input"],
)

# Load results
results_df = catalog.load("inference_predictions_df")
print(f"\n✓ Processed {len(results_df)} queries\n")
results_df

## Step 7: Analyze Results

In [ ]:
# Display summary for each query
for idx, row in results_df.iterrows():
    print(f"\n{'='*80}")
    print(f"Query {row['query_id']}: {row['query']}")
    print(f"{'='*80}")
    print(f"  Prediction: {row['predicted_code']} - {row['predicted_label']}")
    print(f"  Level: {row['predicted_level']}, Score: {row['score']:.3f}")
    print(f"  Ambiguous: {row['ambiguous']}")
    print(f"  Stopping Reason: {row['stopping_reason']}")
    if row['alternatives']:
        print(f"  Alternatives: {len(row['alternatives'])} options")

## Step 8: Stopping Reason Distribution

In [ ]:
# Count stopping reasons
stopping_reason_counts = results_df['stopping_reason'].value_counts()

print("Stopping Reason Distribution:")
print("=" * 80)
for reason, count in stopping_reason_counts.items():
    print(f"  {reason}: {count}")

# Plot if available
try:
    import matplotlib.pyplot as plt
    stopping_reason_counts.plot(kind='bar', title='Stopping Reason Distribution')
    plt.ylabel('Count')
    plt.xlabel('Stopping Reason')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("\n(Install matplotlib for visualization)")

## Step 9: Level Distribution

In [ ]:
# Count predicted levels
level_counts = results_df['predicted_level'].value_counts().sort_index()

print("Level Distribution:")
print("=" * 80)
for level, count in level_counts.items():
    print(f"  Level {level}: {count}")

# Plot if available
try:
    import matplotlib.pyplot as plt
    level_counts.plot(kind='bar', title='Predicted Level Distribution')
    plt.ylabel('Count')
    plt.xlabel('Level')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("\n(Install matplotlib for visualization)")

## Step 10: Export Results

In [ ]:
# Save predictions to CSV
output_path = project_path / "data" / "07_model_output" / "inference_demo_batch.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Create simplified version for export
export_df = results_df[[
    'query_id', 'query', 'predicted_code', 'predicted_label',
    'predicted_level', 'score', 'ambiguous', 'stopping_reason'
]].copy()

export_df.to_csv(output_path, index=False)

print(f"Results saved to: {output_path}")
print(f"  - {len(export_df)} predictions")
print(f"  - Columns: {list(export_df.columns)}")

## Summary

This notebook demonstrated:

1. **Node-by-Node Testing**: Verified each pipeline node works correctly
2. **Single Query**: Tested full pipeline with one query
3. **Batch Inference**: Processed multiple queries efficiently
4. **Routing Analysis**: Examined decision traces and stopping criteria
5. **Result Visualization**: Analyzed stopping reasons and level distributions

## Key Findings

- Root nodes use `"__root__"` as parentCode
- Label-based retrieval + sibling expansion works correctly
- Asymmetric stopping (parent veto + sibling separation) functions as designed
- Batch processing loads model once and processes all queries efficiently

## Next Steps

1. **Parameter Tuning**: Adjust `min_descent_gap` and `parent_veto_margin`
2. **Validation**: Compare predictions against ground truth
3. **Module 3**: Implement incremental learning with evidence updates